In [ ]:
from Tools_U1_plaquette import *

In [ ]:
# ---------------- Parameters λ/m and λ/g ----------------
λ_m = 50.0 #300#100#50.0 
λ_g = 50.0 #300#100#50.0 

N = 13
N_θ = 10001

theta_list = [np.pi/4, np.pi/2, 3*np.pi/4, np.pi]

dt = 0.5
t_max = 1000.0
times = np.arange(0, t_max + dt, dt)

# ---------------- Build Hamiltonian and diagonalize ----------------
(
    E_ν_vector,
    exp_plusiθ_op_matrix,
    exp_minusiθ_op_matrix,
    nθ_op_matrix,
    nθ_sqr_op_matrix
) = single_link_data(λ_m, λ_g, N, N_θ)
H_0_l = sp.diags(E_ν_vector, offsets=0, format='csr')
id_N = sp.eye(N, format='csr')
H_0 = (
    kron_n(H_0_l, id_N, id_N, id_N) +
    kron_n(id_N, H_0_l, id_N, id_N) +
    kron_n(id_N, id_N, H_0_l, id_N) +
    kron_n(id_N, id_N, id_N, H_0_l)
)
H_additional = (2.0 / λ_m) * (
    kron_n(nθ_op_matrix, id_N, nθ_op_matrix, id_N)
  + kron_n(id_N, nθ_op_matrix, id_N, nθ_op_matrix)
  - kron_n(nθ_op_matrix, nθ_op_matrix, id_N, id_N)
  - kron_n(id_N, id_N, nθ_op_matrix, nθ_op_matrix)
)
H_mathieu = H_0 + H_additional

E0, ψ0 = spla.eigsh(H_mathieu, k=1, which="SA", tol=1e-10)
ψ0 = ψ0[:, 0]

# ---------------- Build operators ----------------
vortex_operator = (
    kron_n(nθ_op_matrix, id_N, id_N, id_N) +
    kron_n(id_N, nθ_op_matrix, id_N, id_N) -
    kron_n(id_N, id_N, nθ_op_matrix, id_N) -
    kron_n(id_N, id_N, id_N, nθ_op_matrix)
) / 4
P_op = 0.5 * (
    kron_n(exp_plusiθ_op_matrix, exp_plusiθ_op_matrix,
           exp_minusiθ_op_matrix, exp_minusiθ_op_matrix)
  + kron_n(exp_minusiθ_op_matrix, exp_minusiθ_op_matrix,
           exp_plusiθ_op_matrix, exp_plusiθ_op_matrix)
)
n_sqr_tot = kron_n(nθ_sqr_op_matrix, id_N, id_N, id_N)

# ---------------- Storage ----------------
Fidelity_dict = {}
P_op_dict = {}
n_sqr_dict = {}

# ---------------- Loop ----------------
for theta in theta_list:
    print(f"\n=== Computing for theta = {theta} ===")
    
    vortex = spla.expm_multiply(1j * theta * vortex_operator, ψ0)
    vortex /= np.linalg.norm(vortex)

    Fidelity_t = []
    P_op_t = []
    n_sqr_t = []

    ψ_t = vortex.copy()

    for i, t in enumerate(times):
        print(f"i = {i}, times = {t}")
        if i > 0:
            ψ_t = spla.expm_multiply(-1j * H_mathieu * dt, ψ_t)

        Fidelity_t.append(np.abs(np.vdot(ψ_t, vortex))**2)
        P_op_t.append(np.vdot(ψ_t, P_op @ ψ_t).real)
        n_sqr_t.append(np.vdot(ψ_t, n_sqr_tot @ ψ_t).real)

    Fidelity_dict[theta] = np.array(Fidelity_t)
    P_op_dict[theta] = np.array(P_op_t)
    n_sqr_dict[theta] = np.array(n_sqr_t)

In [ ]:
filename_vortex = f"LGT_SCQs_plaquette_reduced_def_OCT25/Vortex_dynamics_multiple_theta_mathieu_basis_N_{int(N)}_tmax_{int(t_max)}_λ_m_{int(λ_m)}_λ_g_{int(λ_g)}.npz"

Fidelities_arr = np.stack([Fidelity_dict[θ] for θ in theta_list])
P_ops_arr      = np.stack([P_op_dict[θ] for θ in theta_list])
n_sqrs_arr     = np.stack([n_sqr_dict[θ] for θ in theta_list])

np.savez_compressed(
    filename_vortex,
    thetas = np.array(theta_list),
    t=times,
    max_time = t_max,
    Fidelities=Fidelities_arr,
    P_ops=P_ops_arr,
    n_sqrs=n_sqrs_arr
)
print(f"\nResults saved to {filename_vortex}")